# Tools
## 创建工具
使用 @tools 装饰器，
函数的文档字符串会成为工具的描述，帮助模型理解何时使用该工具。

类型提示是必需的，因为它们定义了工具的输入模式。文档字符串应简洁明了，以便模型理解工具的用途

In [1]:
from langchain.tools import tool


@tool
def search_database(query: str, limit: int = 10) -> str:
    """Search the customer database for records matching the query.

    Args:
        query: Search terms to look for
        limit: Maximum number of results to return
    """
    return f"Found {limit} results for '{query}'"

e:\code\LangChainDemo\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


### 自定义工具属性
#### 工具名称
默认情况下，工具名称来源于函数名称。如果需要更具描述性的名称，请对其进行覆盖：

In [ ]:
@tool("web_search")  # Custom name
def search(query: str) -> str:
    """Search the web for information."""
    return f"Results for: {query}"

print(search.name)  # web_search

#### 工具描述
覆盖自动生成的工具描述，以便获得更清晰的模型指导

In [ ]:
@tool(
    "calculator",
    description="Performs arithmetic calculations. Use this for any math problems.",
)
def calc(expression: str) -> str:
    """Evaluate mathematical expressions."""
    return str(eval(expression))

### 高级模式定义
#### Pydantic


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal


class WeatherInput(BaseModel):
    """Input for weather queries"""

    location: str = Field(description="City name or coordinates")
    units: Literal["celsius", "fahrenheit"] = Field(
        default="celsius", description="Temperature unit preference"
    ) # Literal 限制变量的取值范围
    include_forecast: bool = Field(default=False, description="Include 5-day forecast")


@tool(args_schema=WeatherInput)
def get_weather(
    location: str, units: str = "celsius", include_forecast: bool = False
) -> str:
    """Get current weather and optional forecast."""
    temp = 22 if units == "celsius" else 72
    result = f"Current weather in {location}: {temp} degrees {units[0].upper()}"
    if include_forecast:
        result += "\nNext 5 days: Sunny"
    return result

### 保留参数名称
以下参数名称为保留名称，不能用作工具参数。使用这些名称会导致运行时错误。
参数名称	  目的
config	    保留供RunnableConfig内部工具使用
runtime	    保留用于ToolRuntime参数（访问状态、上下文、存储）

## 访问上下文
工具在能够访问运行时信息（例如对话历史记录、用户数据和持久内存）时，其功能最为强大。
是 LangChain 最新工具风格，专门让 工具（tool）能读取对话历史、状态、用户信息。

* 用途
之前写的工具都是独立函数,但现在这个工具 能看见整个聊天的上下文！
给工具开了一个上帝视角，能看到：
    聊天历史 messages
    全局状态 state
    用户信息
    配置

### 短期记忆 State
整个对话的状态仓库
里面存着：messages 消息列表 user_preferences 用户偏好 你想存啥就存啥

#### 访问状态 State


In [7]:
from langchain.messages import HumanMessage
from langchain.tools import ToolRuntime, tool


# ----------------------
# 工具 1：获取用户最后一句话
# ----------------------
@tool
def get_last_user_message(runtime: ToolRuntime) -> str:
    """Get the most recent message from the user."""
    messages = runtime.state["messages"]
    # 从聊天记录里，倒着往上翻，找到最近一句用户说的话，返回给 AI。
    for message in reversed(messages):
        if isinstance(message, HumanMessage):
            return str(message.content)

    return "No user messages found"


# ----------------------
# 工具 2：获取用户偏好
# ----------------------
@tool
def get_user_preference(pref_name: str, runtime: ToolRuntime) -> str:
    """Get a user preference value."""
    # 从全局状态里，拿出用户保存的偏好设置（比如名字、喜欢颜色、时区）
    preferences = runtime.state.get("user_preferences", {})
    return preferences.get(pref_name, "Not set")

ToolRuntime 是 Agent 内部上下文，是由系统自动注入的，不是手动维护的

In [ ]:
# ----------------------
# 初始化本地 Ollama 模型
# ----------------------
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="qwen2.5:3b-instruct-q4_K_M", model_provider="ollama", temperature=0.1
)

# 创建 Agent
agent = create_agent(model=model, tools=[get_last_user_message, get_user_preference])

# 访问模型
response = agent.invoke(
    {"messages": [HumanMessage(content="我喜欢蓝色，我叫小明，我刚才说我喜欢什么？")]},
    config={
        "metadata": {
            "state": {
                "messages": [
                    HumanMessage(content="我喜欢蓝色，我叫小明，我刚才说我喜欢什么？")
                ],
                "user_preferences": {"name": "小明", "favorite_color": "blue"},
            }
        }
    },
)

for msg in response["messages"]:
    print(msg.content)

我喜欢蓝色，我叫小明，我刚才说我喜欢什么？

我喜欢蓝色，我叫小明，我刚才说我喜欢什么？
根据我们之前的对话记录，您说你喜欢蓝色。请问接下来有什么我可以帮助您的吗？如果您有任何问题或需要进一步的信息，请告诉我！


#### 更新状态 State
用于 Command 更新代理的状态。
Command = 给 **LangGraph** 引擎的“指令”
* Command 常见能力
1. 更新状态（你现在用的） Command(update={...}) <=> state.update(...)
2. 控制流程 Command(goto="next_node")
3. 多种组合 Command(
    update={...},
    goto="next_step"
)

* 示例完整链路（LangGraph 完整执行流程）
用户输入：
“我叫小红”

↓
LLM 判断：
需要调用 set_user_name

↓
执行 Tool：
return Command(update={"user_name": "小红"})

↓
LangGraph 引擎：
state["user_name"] = "小红"   ✅（关键！）

↓
后续节点 / Tool：
可以直接用这个新状态

* 用于LangGraph

LangChain
LLM + Tools = 推理

👉 工具只是辅助回答

🧠 LangGraph
LLM + Tools + State + Flow = 应用系统

👉 Tool 可以：

改状态
控流程
驱动整个应用

* 思想转变
LLM = 决策器
 Tool = 执行器
State = 数据库
Command = 写操作
Graph = 工作流



In [29]:
from langchain.messages import ToolMessage
from langgraph.types import Command

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model


# ----------------------
# Tool 1：写入用户名（关键：Command）
# LangChain 新版本
# Tool return → 自动转 ToolMessage
# state 修改 → runtime.state 直接改
# ----------------------
@tool
def set_user_name(new_name: str, runtime: ToolRuntime):
    """Set the user's name"""
    print("✅ set_user_name 被调用")
    # 1️⃣ 返回结果（给 LLM）
    result = f"用户名已设置为 {new_name}"

    # 2️⃣ 更新状态（用 Command）
    runtime.state["user_name"] = new_name

    return result

# ----------------------
# Tool 2：读取用户名并打招呼
# ----------------------
@tool
def greet_user(runtime: ToolRuntime) -> str:
    """Greet the user by name."""
    print("✅ greet_user 被调用")
    return f"你好 {runtime.state.get('user_name', '陌生人')}"


model = init_chat_model(
    model="qwen2.5:3b-instruct-q4_K_M", model_provider="ollama", temperature=0.1
)

# 创建 Agent
agent = create_agent(model=model, tools=[set_user_name, greet_user])

state = {}

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="""
请使用工具完成以下任务：

1. 使用 set_user_name 保存名字 Webster
2. 使用 greet_user 返回问候

注意：
最终答案必须直接来自工具返回值。
""")
        ]
    },
    config={"metadata": {"state": state}},
)

print(response["messages"][-1].content)

✅ set_user_name 被调用
✅ greet_user 被调用
✅ greet_user 被调用
✅ greet_user 被调用
✅ greet_user 被调用
✅ greet_user 被调用
It seems that despite setting your name and attempting to greet you, the `greet_user` function is not functioning as expected. Given this situation, I'll proceed with a general greeting since we have already set your name to 'Webster'.

Hello Webster!


##### Command 最新写法
工具 return 字符串 → LLM 能看到
Command 只负责修改状态
新版写法必须分开写

In [31]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command  # 👈 重点！

# ----------------------
# 工具：用 Command 直接修改状态
# ----------------------
@tool
def set_name(name: str, runtime: ToolRuntime):
    """Set user name and update state."""
    print("🔧 工具执行：set_name")
    
    # ✅ 核心：返回 Command，update 里面直接改 state
    return Command(
        update={"user_name": name},  # 👈 直接修改全局 state！
    ), f"用户名已设置：{name}"

# ----------------------
# 工具：读取状态
# ----------------------
@tool
def get_name(runtime: ToolRuntime):
    """Get current user name."""
    print("🔧 工具执行：get_name")
    return f"当前名字：{runtime.state.get('user_name', '无')}"

# ----------------------
# 模型
# ----------------------
model = init_chat_model(
    model="qwen2.5:3b-instruct-q4_K_M",
    model_provider="ollama"
)

# ----------------------
# 创建 Agent（必须用 create_react_agent）
# ----------------------
agent = create_agent(model, [set_name, get_name])

# ----------------------
# 运行
# ----------------------
response = agent.invoke({
    "messages": [
        HumanMessage(content="把名字设置成 Webster，然后告诉我当前名字")
    ]
})

# ----------------------
# 输出最终结果
# ----------------------
print("\n最终回答：")
print(response["messages"][-1].content)

🔧 工具执行：set_name
🔧 工具执行：get_name

最终回答：
你的名字已经成功设置为 Webster，现在告诉你当前的名字是 Webster。如果你有任何其他问题或需要进一步的帮助，请告诉我！


In [ ]:

# ----------------------
# 2. 模型
# ----------------------
from langchain.chat_models import init_chat_model
from langchain import hub
from langgraph.prebuilt import create_react_agent, AgentExecutor

model = init_chat_model(
    model="qwen2.5:3b-instruct-q4_K_M",
    model_provider="ollama"
)

# ----------------------
# 3. 构建 Agent
# ----------------------
tools = [get_last_user_message, get_user_preference]
prompt = hub.pull("hwchase17/react")
agent = create_react_agent(model, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# ----------------------
# 4. 运行（带状态！）
# ----------------------
result = agent_executor.invoke({
    "messages": [
        HumanMessage(content="我最喜欢的颜色是蓝色，我叫小明")
    ],
    "user_preferences": {
        "name": "小明",
        "favorite_color": "blue"
    }
})

print(result["output"])